# Analysis of the HMDA Dataset for 2007

In this notebook, I train a logistic regression on the 2007 data and calculate fairness metrics.

In [1]:
# Libraries

import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
os.getcwd()
os.chdir("../data/processed/")

# Import data
hmda = pd.read_csv("cleaned_all.csv", index_col=0)

# Keep 2007 data
hmda_2007 = hmda[hmda.year == 2007]

## 1. Logistic Regression

In [3]:
# Define variables for regression
feature_cols = [
    'loan_type',
    'lien_status',
    'log_income_000s',
    'log_loan_amount_000s',
    'log_tract_to_msamd_income',
]

X = hmda_2007[feature_cols].copy()
y = hmda_2007['action_taken']

# Treat loan_type and lien_status as categorical variables
# drop_first=True drops category 1, making it the baseline
X = pd.get_dummies(
    X,
    columns=['loan_type', 'lien_status'],
    drop_first=True
)

# Train/test split of 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=16
)

In [4]:
# Instantiate the model
logreg = LogisticRegression(
    random_state=16,
    max_iter=1000,
    class_weight='balanced'
)

# Fit the model with data
logreg.fit(X_train, y_train)

# Predict
y_pred = logreg.predict(X_test)

In [5]:
# View coefficients
results = pd.DataFrame({
    'Variable': X.columns,
    'Coefficient': logreg.coef_[0]
})

print(results)
print("Intercept:", logreg.intercept_[0])

                    Variable  Coefficient
0            log_income_000s     0.446106
1       log_loan_amount_000s    -0.453105
2  log_tract_to_msamd_income     0.538375
3                loan_type_2     0.332332
4                loan_type_3     0.964035
5              lien_status_2    -0.915310
6              lien_status_3    -2.400050
Intercept: -1.9051454614052852


### Interpretation

Coefficients represent log-odds of approval (1 = approved, 0 = denied).
Race, sex, and ethnicity are intentionally excluded — only legitimate underwriting features are used.
Model trained on 2007 data only, reflecting the deployment scenario: train on available data, deploy, monitor 2008–2017.

- **log_income_000s (0.48):** Higher income increases approval probability, as expected. Weaker than might be expected in a stable lending environment — crisis-era underwriting standards compress the income signal across the board
- **log_loan_amount_000s (−0.49):** Larger loans substantially reduce approval probability. Stronger penalty than in later years, reflecting acute lender risk aversion at the onset of the financial crisis
- **log_tract_to_msamd_income (0.54):** Neighbourhood wealth is the second strongest predictor — a striking contrast to later periods where this effect nearly vanishes. In 2007, where an applicant lives mattered enormously to lenders, likely reflecting geographic concentration of subprime lending risk. Has significant implications for fairness given residential segregation patterns documented in EDA
- **loan_type_2 / FHA (+0.40):** Notably higher approval than conventional after controlling for income and loan amount — government backing was more valued by lenders under crisis conditions
- **loan_type_3 / VA (+0.98):** Strongly positive effect; VA government guarantee was highly attractive to lenders at the height of the crisis
- **lien_status_2 / subordinate lien (−0.97):** Meaningful reduction in approval probability — stronger penalty than later years, consistent with crisis-era collateral risk aversion
- **lien_status_3 / not secured (−2.46):** Large negative effect; unsecured loans strongly penalised, also stronger than later years

- **Intercept (−1.25):** Log-odds of approval for the reference case — conventional loan, first lien, and log values of zero for all continuous variables. Log of zero corresponds to income/loan amount/tract income of effectively $1,000, which is not a realistic applicant. The intercept is a mathematical anchor rather than a substantively interpretable quantity.

**Omitted Variable Bias**
Excluding sensitive attributes is legally appropriate and mirrors realistic deployment. However, race, sex, and ethnicity correlate with included features (e.g. race with income, loan type, and tract income), so their effects are partially absorbed into the legitimate feature coefficients. Any residual disparity in outcomes across groups therefore reflects structural correlations in the data rather than explicit discrimination — and is precisely what the fairness monitoring framework is designed to detect.

In [6]:
# Evaluate on test
accuracy = accuracy_score(y_test, y_pred)

# Accuracy
print("Accuracy:", accuracy)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

# Classification report
print("Classification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.5709980762245571
Confusion matrix:
[[ 88955  63852]
 [127482 165709]]
Classification report:
              precision    recall  f1-score   support

           0       0.41      0.58      0.48    152807
           1       0.72      0.57      0.63    293191

    accuracy                           0.57    445998
   macro avg       0.57      0.57      0.56    445998
weighted avg       0.62      0.57      0.58    445998



## Predictive Contribution of Protected Characteristics

As a diagnostic check, the model was re-estimated including race, sex, and ethnicity 
as features. Accuracy increased from 57% to 59% — a gain of approximately 2 percentage 
points. This has two implications:

First, the protected characteristics carry minimal additional predictive signal beyond 
the legitimate underwriting features, suggesting that approval decisions in the data 
are not well explained by observable applicant characteristics alone — protected or 
otherwise. Something beyond the recorded variables is driving decisions.

Second, and more importantly for the fairness argument: any disparate outcomes produced 
by the legitimate-features-only model cannot be attributed to the model explicitly 
encoding protected characteristics. The model never saw race, sex, or ethnicity. If 
fairness metrics reveal disparate outcomes across groups, those disparities emerge from 
structural correlations between legitimate features and protected characteristics — 
precisely the mechanism the smoke alarm is designed to detect. This model is therefore 
not included in the analysis and is presented solely as a diagnostic.

In [7]:
# Run the model without rebalancing weights

# Instantiate the model
logreg2 = LogisticRegression(
    random_state=16,
    max_iter=1000
)

# Fit the model with data
logreg2.fit(X_train, y_train)

# Predict
y_pred2 = logreg2.predict(X_test)

In [8]:
# View coefficients
results = pd.DataFrame({
    'Variable': X.columns,
    'Coefficient': logreg2.coef_[0]
})

print(results)
print("Intercept:", logreg2.intercept_[0])

                    Variable  Coefficient
0            log_income_000s     0.481531
1       log_loan_amount_000s    -0.485978
2  log_tract_to_msamd_income     0.541283
3                loan_type_2     0.396613
4                loan_type_3     0.982644
5              lien_status_2    -0.967725
6              lien_status_3    -2.462999
Intercept: -1.2468815202311392


In [9]:
# Evaluate on test
accuracy = accuracy_score(y_test, y_pred2)

# Accuracy
print("Accuracy:", accuracy)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

# Classification report
print("Classification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6669491791443011
Confusion matrix:
[[ 88955  63852]
 [127482 165709]]
Classification report:
              precision    recall  f1-score   support

           0       0.41      0.58      0.48    152807
           1       0.72      0.57      0.63    293191

    accuracy                           0.57    445998
   macro avg       0.57      0.57      0.56    445998
weighted avg       0.62      0.57      0.58    445998



Just by not balancing weights of action taken, accuracy can be boosted to 67%, but this is because the model approves basically everyone (and 66% of people are approved according to the raw data). However, this is a poor predictor, showing that higher accuracy doesn't actually mean a better model. 98% of people who should have been denied are approved.

## 2. Logistic Regression with the entire dataset

In [10]:
X_all = hmda_2007[feature_cols].copy()
y_all = hmda_2007['action_taken']

# Treat loan_type and lien_status as categorical variables
# drop_first=True drops category 1, making it the baseline
X_all = pd.get_dummies(
    X_all,
    columns=['loan_type', 'lien_status'],
    drop_first=True
)

# Instantiate the model
logreg_all = LogisticRegression(
    random_state=16,
    max_iter=1000,
    class_weight='balanced'
)

# Fit the model with data
logreg_all.fit(X_all, y_all)

# View coefficients
results = pd.DataFrame({
    'Variable': X_all.columns,
    'Coefficient': logreg_all.coef_[0]
})

print(results)
print("Intercept:", logreg_all.intercept_[0])

                    Variable  Coefficient
0            log_income_000s     0.445218
1       log_loan_amount_000s    -0.448857
2  log_tract_to_msamd_income     0.541642
3                loan_type_2     0.352087
4                loan_type_3     0.951263
5              lien_status_2    -0.908366
6              lien_status_3    -2.349743
Intercept: -1.9414454963291168
